# ByteEmbed — Full SOTA Push (A100)

Runs the three experiments the 12 GB laptop can't:
1. **Scaling curve** — byt5 small → base → **large** (1.2B), one teacher (mE5-base).
2. **Flagship SOTA** — byt5-large distilled hard from the stronger **mE5-large** teacher (big data, 40k steps, 24 langs) to close the gap.
3. **MIRACL** — real passage retrieval (nDCG@10 / recall@100), not just bitext mining.

Everything is **resumable**: results save after each model, the flagship checkpoints its model+optimizer, and re-running a cell skips finished work. **Run the cells top to bottom; do the smoke cell first.**


### 1. GPU check  — confirm you're on an A100 (Runtime → Change runtime type → A100)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')


### 2. Clone repo + install deps
Imports work from the repo root even if the editable install is skipped.


In [ ]:
import os
os.chdir('/content')
REPO = 'https://github.com/Aarushvinod/embedding-research.git'
if not os.path.isdir('/content/embedding-research'):
    !git clone -q $REPO
os.chdir('/content/embedding-research')
!git pull -q
!pip install -q -r requirements-cloud.txt
!pip install -q -e . || echo '(editable install skipped — running from repo root is fine)'
print('setup done | cwd', os.getcwd())


### 3. (Recommended) Persist results + checkpoints to Drive
So a Colab disconnect doesn't lose the long flagship run. **Skip this cell** if you don't want Drive — runs still work, but checkpoints live on ephemeral disk.

_MIRACL loaded without auth in testing; if you hit a rate limit, also run `from huggingface_hub import login; login()`._


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
PERSIST = '/content/drive/MyDrive/byteembed'
for d in ('results', 'checkpoints'):
    os.makedirs(f'{PERSIST}/{d}', exist_ok=True)
    if not os.path.islink(d):
        if os.path.isdir(d): shutil.rmtree(d)
        os.symlink(f'{PERSIST}/{d}', d)
print('persisting results/ and checkpoints/ to', PERSIST)


### 4. Smoke test (~5 min) — validate the whole pipeline on THIS A100 first
Tiny byte-small + byte-base + a baseline + tiny MIRACL. If this prints a table, everything (train → eval → MIRACL → save) works. Then run the real cells below.


In [ ]:
from byte_embed.run_cloud import run
_ = run(smoke=True, out='results/byte_cloud_smoke.json')


### 5. Scaling curve — byt5 small → base → large (mE5-base teacher)
~3–5 h on an A100. Resumable: re-run to skip finished sizes. Tune `scaling_steps` up for stronger models.


In [ ]:
from byte_embed.run_cloud import run
_ = run(phase='scaling', out='results/byte_cloud.json', scaling_steps=15000)


### 6. Flagship SOTA model — byt5-large ← mE5-large teacher
The push to close the gap: stronger teacher, 40k steps, 24 langs, **checkpointed** (disconnect-safe — just re-run this cell to resume). ~6–8 h. Set `miracl_extra=20000` for a harder, more realistic retrieval pool.


In [ ]:
from byte_embed.run_cloud import run
_ = run(phase='flagship', out='results/byte_cloud.json', flagship_steps=40000,
        miracl_q=250, miracl_extra=0)


### 7. Results — table, scaling curve, SOTA gap, figure


In [ ]:
import json
from byte_embed.run_cloud import _summary
res = json.load(open('results/byte_cloud.json'))
_summary(res)


In [ ]:
# quality-vs-params figure: MIRACL + Tatoeba for byte models, baselines as stars
import matplotlib.pyplot as plt
M = res['models']
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, key, title in [(axes[0],'miracl','MIRACL nDCG@10'),(axes[1],'tatoeba_mean','Tatoeba')]:
    bx, by, names = [], [], []
    for n, r in M.items():
        y = (r.get('miracl') or {}).get('ndcg@10_mean') if key=='miracl' else r.get(key)
        if y is None or not r.get('params'): continue
        p = r['params']/1e6
        if r.get('kind')=='baseline':
            ax.scatter(p, y, marker='*', s=200, c='#c33', zorder=3)
            ax.annotate(n, (p, y), fontsize=8, xytext=(4,4), textcoords='offset points')
        else:
            bx.append(p); by.append(y); names.append(n)
    order = sorted(range(len(bx)), key=lambda i: bx[i])
    ax.plot([bx[i] for i in order], [by[i] for i in order], 'o-', c='#2a7', label='byte student')
    ax.set_xlabel('parameters (M)'); ax.set_ylabel(title); ax.legend(fontsize=9)
fig.suptitle('Byte students vs strong baselines (★)'); fig.tight_layout()
fig.savefig('results/fig_cloud.png', dpi=120); plt.show()


### 8. Download results
Already on Drive if you ran cell 3. Otherwise grab the JSON + figure here.


In [ ]:
from google.colab import files
files.download('results/byte_cloud.json')
files.download('results/fig_cloud.png')
